# Janessa Leone — MMM Modeling Notebook

**Client:** Janessa Leone (e-commerce)  
**KPI:** Revenue  
**Geo:** National (single series — no geo dimension)  
**Channels:** Brand, Non-Brand, Brand Shopping, Prospecting, Retargeting, Remarketing, Pinterest, ShopMy (affiliate), AWIN (affiliate)  
**Affiliate volume metric:** Clicks (not impressions) — stored in `*_Impressions` columns per pipeline convention  

---
**Workflow:**
1. Iterate here locally (dev mode — 1 chain, 200 samples)
2. When happy with spec: run the *Save settings* cell at the bottom → `configs/Janessa_Leone.yaml`
3. `git push`
4. Open `notebooks/colab_runner.ipynb` → set `CLIENT = "janessa_leone"` → Run All

### Packages

In [ ]:
import sys
print(sys.executable)

In [ ]:
# ── Environment ────────────────────────────────────────────────────────────────
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU-only — Meridian runs on CPU via MCMC

import re
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
tfd = tfp.distributions

from meridian.data import data_frame_input_data_builder
from meridian.model import model, spec
from meridian.model import prior_distribution
from meridian.analysis import visualizer, analyzer, summarizer
from meridian.analysis import optimizer
import arviz as az

import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd().parents[2]))
from src import reviewer

print(f"TF: {tf.__version__}  |  TFP: {tfp.__version__}  |  az: {az.__version__}")

### Load Data

In [ ]:
from pathlib import Path

# Locate processed data dir — works from workspace root or notebook directory
_proc = Path.cwd() / "data/processed/janessa_leone"
if not _proc.exists():
    _proc = Path.cwd().parents[2] / "data/processed/janessa_leone"

_months = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
           "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}

def _vintage_key(p):
    m = re.search(r"_([A-Za-z]{3})(\d{2})", p.stem, re.IGNORECASE)
    if not m: return (0, 0)
    return (2000 + int(m.group(2)), _months.get(m.group(1).lower(), 0))

_files = sorted(
    [f for f in _proc.glob("JL_mmm_data_*.csv") if "test" not in f.name],
    key=_vintage_key, reverse=True
)
assert _files, f"No JL_mmm_data_*.csv found in {_proc}"
print(f"Loading: {_files[0].name}")
raw_data = pd.read_csv(_files[0], parse_dates=["date"])
print(f"{len(raw_data)} rows | {raw_data['date'].min().date()} → {raw_data['date'].max().date()}")
print(f"Columns: {list(raw_data.columns)}")

In [ ]:
raw_data.head()

### Date Filter

Adjust `start_date` / `end_date` as needed. The first few weeks of 2024 may be ramp-up — inspect and trim if sparse.

In [ ]:
start_date = pd.to_datetime("2024-01-01")
end_date   = raw_data["date"].max()  # use full history by default

raw_data["date"] = pd.to_datetime(raw_data["date"])
raw_data = raw_data[(raw_data["date"] >= start_date) & (raw_data["date"] <= end_date)].copy()
raw_data = raw_data.sort_values("date").reset_index(drop=True)

print(f"Filtered: {len(raw_data)} rows | {raw_data['date'].min().date()} → {raw_data['date'].max().date()}")

### Channels + float32 Casting

Note: ShopMy and AWIN `_Impressions` columns contain **Clicks** (not ad impressions).  
Meridian treats them as the volume metric regardless of name — this is correct for affiliate channels.

In [ ]:
# ── Channel lists ──────────────────────────────────────────────────────────────
channels = [
    "Brand", "Non_Brand", "Brand_Shopping",
    "Prospecting", "Retargeting", "Remarketing",
    "Pinterest", "ShopMy", "AWIN",
]

# No organic channels in v1 — Social Organic TBD

# ── float32 casting ────────────────────────────────────────────────────────────
raw_data["Revenue"]      = raw_data["Revenue"].astype(np.float32)
raw_data["black_friday"] = raw_data["black_friday"].astype(np.float32)

for c in channels:
    raw_data[f"{c}_Cost"]        = raw_data[f"{c}_Cost"].astype(np.float32)
    raw_data[f"{c}_Impressions"] = raw_data[f"{c}_Impressions"].astype(np.float32)

# Quick spend sanity check
total_spend = sum(raw_data[f"{c}_Cost"].sum() for c in channels)
print(f"Total spend across all channels: ${total_spend:,.0f}")
for c in channels:
    t = raw_data[f"{c}_Cost"].sum()
    print(f"  {c:<18} ${t:>10,.0f}  ({t/total_spend*100:.1f}%)")

### Build Input Data

In [ ]:
# ── Input Data Builder (national — no geo, no population) ─────────────────────
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(kpi_type="revenue")

builder = builder.with_kpi(
    raw_data,
    kpi_col="Revenue",
    time_col="date",
)
builder = builder.with_media(
    raw_data,
    media_channels=channels,
    media_spend_cols=[f"{c}_Cost" for c in channels],
    media_cols=[f"{c}_Impressions" for c in channels],
    time_col="date",
)
builder = builder.with_controls(
    raw_data,
    control_cols=["black_friday"],
    time_col="date",
)

input_data = builder.build()
print("Input data built successfully.")

### Build Priors

ROI ranges below are **placeholders** from the skeleton config.  
Update after reviewing last-touch attribution data and first dev run posteriors.

**Prior archetype notes:**
- Brand / Non_Brand / Brand_Shopping: paid search — demand capture, typically higher ROI
- Prospecting: broad reach — lower incremental lift expected; widen range if uncertain
- Retargeting / Remarketing: warm audiences — moderate-high ROI; both sparse, watch posteriors
- Pinterest: new channel, very sparse — wide range to avoid over-constraining
- ShopMy / AWIN: affiliate / performance — volume metric is Clicks; ROI plausibility varies by commission structure

In [ ]:
# ==========================================================
# ROI PRIOR STRATEGY
# Each tuple = (low, high) 95% credible interval for that channel's ROI.
# ⚠ PLACEHOLDERS — update from LT analysis before prod run.
# ==========================================================
roi_ranges = {
    "Brand":         (1.5,  9.0),   # placeholder — paid search branded
    "Non_Brand":     (1.5,  8.0),   # placeholder — paid search generic
    "Brand_Shopping":(2.0,  9.0),   # placeholder — shopping branded; high purchase intent
    "Prospecting":   (0.5,  4.0),   # placeholder — broad reach; lower incremental expected
    "Retargeting":   (1.5,  8.0),   # placeholder — warm audience; sparse data
    "Remarketing":   (1.5,  8.0),   # placeholder — warm audience; sparse data
    "Pinterest":     (0.5,  4.0),   # placeholder — new channel; wide range intentional
    "ShopMy":        (0.5,  5.0),   # placeholder — affiliate/influencer; commission-based
    "AWIN":          (0.5,  5.0),   # placeholder — affiliate; started Q3 2025
}

roi_dists = [
    prior_distribution.lognormal_dist_from_range(
        low=roi_ranges[c][0],
        high=roi_ranges[c][1],
        mass_percent=0.95,
    )
    for c in channels
]

roi_loc_vec   = tf.cast([d.loc   for d in roi_dists], tf.float32)
roi_scale_vec = tf.cast([d.scale for d in roi_dists], tf.float32)

my_priors = prior_distribution.PriorDistribution(
    roi_m=tfd.LogNormal(
        loc=roi_loc_vec,
        scale=roi_scale_vec,
    )
)

# Print implied prior medians for quick sanity check
print("Prior medians (exp(loc)):")
for c, d in zip(channels, roi_dists):
    print(f"  {c:<18}  median={float(tf.exp(d.loc)):.2f}x  scale={float(d.scale):.3f}")

### Model Spec

In [ ]:
# ==========================================================
# MODEL SPEC
# knots=26 — one every ~4 weeks for a 2.5-year series; flexible but not overfit
# max_lag=6 — standard for e-commerce; reduce to 4 if digital-heavy mix warrants it
# ==========================================================
model_spec = spec.ModelSpec(
    prior=my_priors,
    media_prior_type="roi",
    knots=26,
    max_lag=6,
    adstock_decay_spec="geometric",
    media_effects_dist="log_normal",
)

mmm = model.Meridian(input_data=input_data, model_spec=model_spec)
print("Model initialized.")

### Run Model

In [ ]:
# DEV run — fast iteration (~5 min)
# Swap n_chains=4, n_adapt/burnin/keep=500 when sending to Colab for prod
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=1,
    n_adapt=200,
    n_burnin=200,
    n_keep=200,
    seed=42,
)

### Review Model

In [ ]:
reviewer.ModelReviewer(mmm).run()

### Core Health Check

**r_hat** < 1.05 = good convergence. **ess_bulk** > 100 per chain = adequate sampling.  
Flag any channel where posterior is nearly identical to prior — indicates non-identification.

In [ ]:
idata = mmm.inference_data
az.summary(
    idata,
    var_names=[
        "beta_m",
        "alpha_m",
        "ec_m",
        "slope_m",
        "roi_m",
    ],
    round_to=3,
)

### Model Fit Diagnostics

In [ ]:
m_analyzer = analyzer.Analyzer(mmm)
summary_df = m_analyzer.summary_metrics()
summary_df

In [ ]:
summary_df.tail(30)

### Model Output

In [ ]:
from meridian.analysis import summarizer

summary = summarizer.Summarizer(mmm)
contributions = summary.contribution_breakdown()
contributions

In [ ]:
media_summary = visualizer.MediaSummary(mmm)
media_summary.plot_contribution_pie_chart()

In [ ]:
vis = visualizer.MediaEffects(mmm)
vis.plot_response_curves()

In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

### Full Model Output HTML

In [ ]:
import IPython

start_str = raw_data["date"].min().strftime("%Y-%m-%d")
end_str   = raw_data["date"].max().strftime("%Y-%m-%d")

mmm.output_model_results(
    start_date=start_str,
    end_date=end_str,
    filename="janessa_leone_summary_output.html",
    output_directory="./",
)
IPython.display.HTML(filename="janessa_leone_summary_output.html")

### Budget Optimizer

In [ ]:
budget_optimizer = optimizer.BudgetOptimizer(mmm)

In [ ]:
optimization_results = budget_optimizer.optimize(
    fixed_budget=True,
    # start_date="YYYY-MM-DD",  # optional: restrict optimization window
    # end_date="YYYY-MM-DD",
)

optimization_results.output_optimization_summary("janessa_leone_optimization_summary.html", "./")
IPython.display.HTML(filename="janessa_leone_optimization_summary.html")

---
## Save Settings → `configs/Janessa_Leone.yaml`

Run this cell when satisfied with the model spec.  
**Push to GitHub before triggering a Colab run.**

In [ ]:
import yaml
from pathlib import Path

# ── MCMC settings ──────────────────────────────────────────────────────────────
mcmc_dev  = dict(n_chains=1, n_adapt=200, n_burnin=200, n_keep=200)
mcmc_prod = dict(n_chains=4, n_adapt=500, n_burnin=500, n_keep=500)

# ── Locate repo root (works from notebook dir or workspace root) ───────────────
_here = Path.cwd()
_root = _here
for _ in range(5):
    if (_root / "configs").exists():
        break
    _root = _root.parent

# ── Assemble config ────────────────────────────────────────────────────────────
config = {
    "client_id":   "janessa_leone",
    "kpi_column":  "Revenue",
    "kpi_type":    "revenue",
    "date_column": "date",
    # no geo_column — national single-series model
    "data_path":       f"data/processed/janessa_leone/{_files[0].name}",
    "gcs_data_path":   f"gs://mmm-pipeline-results/clients/janessa_leone/{_files[0].name}",
    "output_path":     "outputs/janessa_leone/",
    "gcs_output_path": "gs://mmm-pipeline-results/clients/janessa_leone/runs/",
    "channels": channels,
    "organic_channels": [],  # Social Organic TBD
    "controls": ["black_friday"],
    "prior_type": "roi",
    "prior_roi_ranges": {
        c: {"low": float(lo), "high": float(hi)}
        for c, (lo, hi) in roi_ranges.items()
    },
    "prior_roi_mass_percent": 0.95,
    "model_spec": dict(
        media_prior_type="roi",
        knots=26,
        max_lag=6,
        adstock_decay_spec="geometric",
        media_effects_dist="log_normal",
    ),
    "mcmc": {"dev": mcmc_dev, "prod": mcmc_prod},
    "max_runtime_minutes": 45,
}

config_path = _root / "configs" / "Janessa_Leone.yaml"
with open(config_path, "w") as fh:
    yaml.dump(config, fh, default_flow_style=False, sort_keys=False)

print(f"Config saved → {config_path}")
print("Push to GitHub before running on Colab.")